## Mutation Testing on Mistral LLM

Using the code generation database from the MongoDB, this notebook will run **zero shot, one shot** and **few shot prompts** on a Mistral LLM. Each prompt technique also includes **no mutation, sequential mutated** and **random mutated** programs. In total, 9 experiments are run through this notebook. All logs are stored in csv files automatically for your analysis.

In [1]:
import os
import sys

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from code_generation.code_generation_tester import CodeGenerationTester
from code_generation.prompt_templates.prompt_template import OpenEndedPromptTemplate
from utility.constants import BigCodeBench, HumanEval, LexicalMutations, SyntacticMutations, LogicalMutations, PromptTypes, CodeGeneration, ReasoningModels, NonReasoningModels

In [4]:
## Declaring Prompt Type Constants
ZERO_SHOT = PromptTypes.ZERO_SHOT
ONE_SHOT = PromptTypes.ONE_SHOT
FEW_SHOT = PromptTypes.FEW_SHOT

## Declaring Mutation Constants
RANDOM_MUTATION = LexicalMutations.RANDOM
SEQUENTIAL_MUTATION = LexicalMutations.SEQUENTIAL
LITERAL_FORMAT = LexicalMutations.LITERAL_FORMAT

## Declaring Benchmark Name Constants
BIGCODEBENCH = BigCodeBench.NAME
HUMANEVAL = HumanEval.NAME

## Declaring Reasoning Model Name Constants
GPT5 = ReasoningModels.GPT5['name']

## Declaring Non-Reasoning Model Name Constants
MISTRAL = NonReasoningModels.MISTRAL_SMALL_LATEST['name']

In [5]:
reasoning_models = [getattr(ReasoningModels, model) for model in dir(ReasoningModels) if not model.startswith("_")]
non_reasoning_models = [getattr(NonReasoningModels, model) for model in dir(NonReasoningModels) if not model.startswith("_")]
print('Reasoning models supported by this framework are:')
for idx, model in enumerate(reasoning_models):
    print(f"{idx+1}: '{model['name']}'")
print('=' * 50)
print('Non-reasoning models supported by this framework are:')
for idx, model in enumerate(non_reasoning_models):
    print(f"{idx+1}: '{model['name']}'")

Reasoning models supported by this framework are:
1: 'gpt-4o'
2: 'gpt-5'
Non-reasoning models supported by this framework are:
1: 'mistral-small-latest'


In [6]:
task_set = HUMANEVAL

try:
    llmtester = CodeGenerationTester(f"{task_set}_Code_Generation")
except Exception as e:
    print(f'llmtester could not launch due to the following error: {e}')

MongoDB connected


In [7]:
num_tests = llmtester.question_database.count_documents({})

In [8]:
valid_mutations = CodeGeneration.MUTATIONS
print("These are the valid mutation names for code generation:")
for idx, mutation in enumerate(valid_mutations):
    if mutation != LITERAL_FORMAT:
        print(idx+1, mutation)

These are the valid mutation names for code generation:
2 random
3 sequential


# Run your experiments

In [9]:
# %%script false --no-raise-error
mutations = [RANDOM_MUTATION]
prompt_type = ZERO_SHOT
model_name = MISTRAL

# Forming the results directory
results_dir =os.path.join(proj_dir, f'results/code_generation/{model_name}')
os.makedirs(results_dir, exist_ok=True)

mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"
output_file_path=f"{results_dir}/{task_set}_{prompt_type}_{mutation_str}.csv"

pass_count = llmtester.run_code_generation_test(
    prompt_helper = OpenEndedPromptTemplate().return_appropriate_prompt(prompt_type),
    # num_tests=num_tests,
    num_tests=10,
    mutations = mutations,
    prompt_type= prompt_type,
    output_file_path=output_file_path,
    task_set = task_set,
    model_name= model_name,
)

print(fr"Results saved in {output_file_path}")


  0%|          | 0/10 [00:00<?, ?it/s]

HumanEvalo0: Function failed to run due to following error -> 


 10%|█         | 1/10 [00:07<01:03,  7.06s/it]

HumanEvalo1: Function failed to run due to following error -> 


100%|██████████| 10/10 [01:09<00:00,  6.93s/it]

Results saved in /Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/code_generation/mistral-small-latest/HumanEval_zero_shot_random.csv
